In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
import openpyxl

pd.set_option('display.max_rows', 500)


In [2]:


# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

DATA_URL = (
    "https://www.data.gouv.fr/api/1/datasets/r/8fdb0926-ea9d-4fb4-a136-7767cd97e30b"
)

path_data_raw     = os.path.join(RAW_DIR, "2017_burvot_t1_france_entiere.txt")
path_rhone_bronze = os.path.join(BRONZE_DIR, "2017-bronze_burvot_t1_rhone_69.csv")



In [3]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_data_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(DATA_URL, timeout=180)
    resp.raise_for_status()
    with open(path_data_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_data_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


📥 Downloading source to RAW...
✅ Saved to RAW: ../../../data/raw/2017_burvot_t1_france_entiere.txt


In [4]:
print("\n📊 RAW FILE STATS")

# Taille fichier
file_size_mb = os.path.getsize(path_data_raw) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Nombre de lignes (brut)
with open(path_data_raw, "r", encoding="cp1252", errors="ignore") as f:
    n_lines = sum(1 for _ in f)

print(f"Total lines (including header): {n_lines}")

# Lecture très légère (juste pour aperçu)
df_sample = pd.read_csv(
    path_data_raw,
    sep=";",
    encoding="cp1252",
    nrows=5
)

print("\nColumns detected (raw header):")
print(df_sample.columns.tolist())

print("\nSample preview:")
display(df_sample.head())

# Nombre de colonnes détectées
print(f"\nDetected columns count: {len(df_sample.columns)}")


📊 RAW FILE STATS
File size: 33.14 MB
Total lines (including header): 69243

Columns detected (raw header):
['Code du département', 'Libellé du département', 'Code de la circonscription', 'Libellé de la circonscription', 'Code de la commune', 'Libellé de la commune', 'Code du b.vote', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp']

Sample preview:


Code du département  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1 598  92  15,38 506 84,62 2  0,33 0,40 9 1,51 1,78 495 82,78 97,83 1 M DUPONT-AIGNAN Nicolas 34 5,69 6,87 2 F LE PEN Marine 126 21,07 25,45 3 M MACRON Emmanuel 119 19,90 24,04 4 M HAMON Benoît 29 4,85 5,86 5 F ARTHAUD Nathalie 4 0,67 0,81 6 M POUTOU Philippe 4  0,67 0,81 7 M CHEMINADE Jacques 2 0,33 0,40                    8   
      5 5ème circonscription 2 L'Abergement-de-Varey   1 209  25  11,96 184 88,04 6  2,87 3,26 2 0,96 1,09 176 84,21 95,65 1 M DUPONT-AIGNAN Nicolas 6  2,87 3,41 2 F LE PEN Marine 48  22,97 27,27 3 M MACRON Emmanuel 37  17,70 21,02 4 M HAMON Benoît 13 6,22 7,39 5 F ARTHAUD Nathalie 2 0,96 1,14 6 M POUTOU Philippe 2  0,96 1,14 7 M CHEMINADE Jacques 0 0,00 0,00                    8   
                             4 Ambérieu-en-Bugey       1 1116 233 20,88 883 79,12 17 1,52 1,93 6 0,54 0,68 860 77,06 97,40 1 M DUPONT-AIGNAN Nicolas 58 5,20 6,74 2 F LE PEN Marine 233 20,88 27,09 3 M MACRON Emmanuel 156 13,98 18,14 4 M HAMON Benoît 47 4,21 5,47 5 F ARTHAUD Nathalie 6 0,54 0,70 6 M POUTOU Philippe 15 1,34 1,74 7 M CHEMINADE Jacques 0 0,00 0,00                    8   
                                                       2 1128 256 22,70 872 77,30 19 1,68 2,18 3 0,27 0,34 850 75,35 97,48 1 M DUPONT-AIGNAN Nicolas 42 3,72 4,94 2 F LE PEN Marine 234 20,74 27,53 3 M MACRON Emmanuel 183 16,22 21,53 4 M HAMON Benoît 51 4,52 6,00 5 F ARTHAUD Nathalie 5 0,44 0,59 6 M POUTOU Philippe 16 1,42 1,88 7 M CHEMINADE Jacques 1 0,09 0,12                    8   
                                                       3 1116 227 20,34 889 79,66 11 0,99 1,24 4 0,36 0,45 874 78,32 98,31 1 M DUPONT-AIGNAN Nicolas 47 4,21 5,38 2 F LE PEN Marine 218 19,53 24,94 3 M MACRON Emmanuel 191 17,11 21,85 4 M HAMON Benoît 53 4,75 6,06 5 F ARTHAUD Nathalie 4 0,36 0,46 6 M POUTOU Philippe 12 1,08 1,37 7 M CHEMINADE Jacques 0 0,00 0,00                    8   

                                                                                                                                                                                                                                                                                                                                                                          Libellé du département  \
1 Ain 4 4ème circonscription 1 L'Abergement-Clémenciat 1 598  92  15,38 506 84,62 2  0,33 0,40 9 1,51 1,78 495 82,78 97,83 1 M DUPONT-AIGNAN Nicolas 34 5,69 6,87 2 F LE PEN Marine 126 21,07 25,45 3 M MACRON Emmanuel 119 19,90 24,04 4 M HAMON Benoît 29 4,85 5,86 5 F ARTHAUD Nathalie 4 0,67 0,81 6 M POUTOU Philippe 4  0,67 0,81 7 M CHEMINADE Jacques 2 0,33 0,40                      M   
      5 5ème circonscription 2 L'Abergement-de-Varey   1 209  25  11,96 184 88,04 6  2,87 3,26 2 0,96 1,09 176 84,21 95,65 1 M DUPONT-AIGNAN Nicolas 6  2,87 3,41 2 F LE PEN Marine 48  22,97 27,27 3 M MACRON Emmanuel 37  17,70 21,02 4 M HAMON Benoît 13 6,22 7,39 5 F ARTHAUD Nathalie 2 0,96 1,14 6 M POUTOU Philippe 2  0,96 1,14 7 M CHEMINADE Jacques 0 0,00 0,00                      M   
                             4 Ambérieu-en-Bugey       1 1116 233 20,88 883 79,12 17 1,52 1,93 6 0,54 0,68 860 77,06 97,40 1 M DUPONT-AIGNAN Nicolas 58 5,20 6,74 2 F LE PEN Marine 233 20,88 27,09 3 M MACRON Emmanuel 156 13,98 18,14 4 M HAMON Benoît 47 4,21 5,47 5 F ARTHAUD Nathalie 6 0,54 0,70 6 M POUTOU Philippe 15 1,34 1,74 7 M CHEMINADE Jacques 0 0,00 0,00                      M   
                                                       2 1128 256 22,70 872 77,30 19 1,68 2,18 3 0,27 0,34 850 75,35 97,48 1 M DUPONT-AIGNAN Nicolas 42 3,72 4,94 2 F LE PEN Marine 234 20,74 27,53 3 M MACRON Emmanuel 183 16,22 21,53 4 M HAMON Benoît 51 4,52 6,00 5 F ARTHAUD Nathalie 5 0,44 0,59 6 M POUTOU Philippe 16 1,42 1,88 7 M CHEMINADE Jacques 1 0,09 0,12                      M   
                                                       3 1116 227 20,34 889 79,66 11 0,99 1,24 4 0,36 0,45


Detected columns count: 28


In [5]:
# ── 3. BRONZE LAYER ─────────────────────
print("\nProcessing RAW -> BRONZE...")

COMMON_FIELDS = [
    "Code du département",
    "Libellé du département",
    "Code de la circonscription",
    "Libellé de la circonscription",
    "Code de la commune",
    "Libellé de la commune",
    "Code du b.vote",
    "Inscrits",
    "Abstentions",
    "% Abs/Ins",
    "Votants",
    "% Vot/Ins",
    "Blancs",
    "% Blancs/Ins",
    "% Blancs/Vot",
    "Nuls",
    "% Nuls/Ins",
    "% Nuls/Vot",
    "Exprimés",
    "% Exp/Ins",
    "% Exp/Vot",
]

CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
N_CANDIDATES = 11

all_columns = COMMON_FIELDS.copy()
for k in range(1, N_CANDIDATES + 1):
    for field in CAND_FIELDS:
        all_columns.append(f"{field}_{k}")

df_all = pd.read_csv(
    path_data_raw,
    sep=";",
    dtype=str,
    encoding="cp1252",
    header=None,
    skiprows=1,          # on saute l'en-tête du fichier source
    names=all_columns,   # on impose la vraie structure
    engine="python"
)

# nettoyage
df_all.columns = df_all.columns.str.strip()
df_all["Code du département"] = df_all["Code du département"].astype(str).str.strip().str.zfill(2)

# filtre Rhône
df_bronze = df_all[df_all["Code du département"] == "69"].copy()

# métadonnées bronze
df_bronze["extraction_source_url"] = DATA_URL
df_bronze["ingestion_timestamp"] = datetime.now().isoformat()
df_bronze["source_file_name"] = os.path.basename(path_data_raw)

# sauvegarde
df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")

display(df_bronze.head())
print(f"BRONZE dataset created: {path_rhone_bronze}")
print(f"Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")


Processing RAW -> BRONZE...


,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,...,N°Panneau_11,Sexe_11,Nom_11,Prénom_11,Voix_11,% Voix/Ins_11,% Voix/Exp_11,extraction_source_url,ingestion_timestamp,source_file_name
47091,69,Rhône,08,8ème circonscription,001,Affoux,0001,259,29,"11,20",...,11,M,FILLON,François,46,"17,76","20,26",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47092,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,224,47,"20,98",...,11,M,FILLON,François,48,"21,43","28,24",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47093,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0001,884,155,"17,53",...,11,M,FILLON,François,189,"21,38","26,51",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47094,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0002,781,162,"20,74",...,11,M,FILLON,François,154,"19,72","25,37",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47095,69,Rhône,09,9ème circonscription,004,Alix,0001,455,82,"18,02",...,11,M,FILLON,François,60,"13,19","16,67",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt


BRONZE dataset created: ../../../data/bronze/2017-bronze_burvot_t1_rhone_69.csv
Rows: 1287 | Columns: 101


In [6]:
print("\n📊 BRONZE DATASET STATS")

# Taille fichier
file_size_mb = os.path.getsize(path_rhone_bronze) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Shape
print(f"Rows: {df_bronze.shape[0]}")
print(f"Columns: {df_bronze.shape[1]}")

# Colonnes
print("\nColumns:")
print(df_bronze.columns.tolist())

# Types
print("\nData types:")
print(df_bronze.dtypes)

# Valeurs manquantes
print("\nMissing values per column:")
print(df_bronze.isna().sum())

# Vérification département
print("\nDepartment distribution:")
print(df_bronze["Code du département"].value_counts())

# Vérification communes (top 10)
print("\nTop 10 communes:")
print(df_bronze["Libellé de la commune"].value_counts().head(10))

# Stats sur une variable clé
print("\nInscrits stats:")
print(df_bronze["Inscrits"].astype(float).describe())


📊 BRONZE DATASET STATS
File size: 0.79 MB
Rows: 1287
Columns: 101

Columns:
['Code du département', 'Libellé du département', 'Code de la circonscription', 'Libellé de la circonscription', 'Code de la commune', 'Libellé de la commune', 'Code du b.vote', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau_1', 'Sexe_1', 'Nom_1', 'Prénom_1', 'Voix_1', '% Voix/Ins_1', '% Voix/Exp_1', 'N°Panneau_2', 'Sexe_2', 'Nom_2', 'Prénom_2', 'Voix_2', '% Voix/Ins_2', '% Voix/Exp_2', 'N°Panneau_3', 'Sexe_3', 'Nom_3', 'Prénom_3', 'Voix_3', '% Voix/Ins_3', '% Voix/Exp_3', 'N°Panneau_4', 'Sexe_4', 'Nom_4', 'Prénom_4', 'Voix_4', '% Voix/Ins_4', '% Voix/Exp_4', 'N°Panneau_5', 'Sexe_5', 'Nom_5', 'Prénom_5', 'Voix_5', '% Voix/Ins_5', '% Voix/Exp_5', 'N°Panneau_6', 'Sexe_6', 'Nom_6', 'Prénom_6', 'Voix_6', '% Voix/Ins_6', '% Voix/Exp_6', 'N°Panneau_7', 'Sexe_7', 'Nom_7', 'Pr

In [7]:
df_bronze.dtypes

pd.set_option('display.max_columns', None)

display(df_bronze.head())


,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau_1,Sexe_1,Nom_1,Prénom_1,Voix_1,% Voix/Ins_1,% Voix/Exp_1,N°Panneau_2,Sexe_2,Nom_2,Prénom_2,Voix_2,% Voix/Ins_2,% Voix/Exp_2,N°Panneau_3,Sexe_3,Nom_3,Prénom_3,Voix_3,% Voix/Ins_3,% Voix/Exp_3,N°Panneau_4,Sexe_4,Nom_4,Prénom_4,Voix_4,% Voix/Ins_4,% Voix/Exp_4,N°Panneau_5,Sexe_5,Nom_5,Prénom_5,Voix_5,% Voix/Ins_5,% Voix/Exp_5,N°Panneau_6,Sexe_6,Nom_6,Prénom_6,Voix_6,% Voix/Ins_6,% Voix/Exp_6,N°Panneau_7,Sexe_7,Nom_7,Prénom_7,Voix_7,% Voix/Ins_7,% Voix/Exp_7,N°Panneau_8,Sexe_8,Nom_8,Prénom_8,Voix_8,% Voix/Ins_8,% Voix/Exp_8,N°Panneau_9,Sexe_9,Nom_9,Prénom_9,Voix_9,% Voix/Ins_9,% Voix/Exp_9,N°Panneau_10,Sexe_10,Nom_10,Prénom_10,Voix_10,% Voix/Ins_10,% Voix/Exp_10,N°Panneau_11,Sexe_11,Nom_11,Prénom_11,Voix_11,% Voix/Ins_11,% Voix/Exp_11,extraction_source_url,ingestion_timestamp,source_file_name
47091,69,Rhône,08,8ème circonscription,001,Affoux,0001,259,29,"11,20",230,"88,80",0,"0,00","0,00",3,"1,16","1,30",227,"87,64","98,70",1,M,DUPONT-AIGNAN,Nicolas,15,"5,79","6,61",2,F,LE PEN,Marine,75,"28,96","33,04",3,M,MACRON,Emmanuel,46,"17,76","20,26",4,M,HAMON,Benoît,7,"2,70","3,08",5,F,ARTHAUD,Nathalie,1,"0,39","0,44",6,M,POUTOU,Philippe,4,"1,54","1,76",7,M,CHEMINADE,Jacques,0,"0,00","0,00",8,M,LASSALLE,Jean,6,"2,32","2,64",9,M,MÉLENCHON,Jean-Luc,24,"9,27","10,57",10,M,ASSELINEAU,François,3,"1,16","1,32",11,M,FILLON,François,46,"17,76","20,26",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47092,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,224,47,"20,98",177,"79,02",4,"1,79","2,26",3,"1,34","1,69",170,"75,89","96,05",1,M,DUPONT-AIGNAN,Nicolas,13,"5,80","7,65",2,F,LE PEN,Marine,50,"22,32","29,41",3,M,MACRON,Emmanuel,34,"15,18","20,00",4,M,HAMON,Benoît,5,"2,23","2,94",5,F,ARTHAUD,Nathalie,0,"0,00","0,00",6,M,POUTOU,Philippe,3,"1,34","1,76",7,M,CHEMINADE,Jacques,1,"0,45","0,59",8,M,LASSALLE,Jean,1,"0,45","0,59",9,M,MÉLENCHON,Jean-Luc,15,"6,70","8,82",10,M,ASSELINEAU,François,0,"0,00","0,00",11,M,FILLON,François,48,"21,43","28,24",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47093,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0001,884,155,"17,53",729,"82,47",15,"1,70","2,06",1,"0,11","0,14",713,"80,66","97,81",1,M,DUPONT-AIGNAN,Nicolas,45,"5,09","6,31",2,F,LE PEN,Marine,99,"11,20","13,88",3,M,MACRON,Emmanuel,190,"21,49","26,65",4,M,HAMON,Benoît,55,"6,22","7,71",5,F,ARTHAUD,Nathalie,2,"0,23","0,28",6,M,POUTOU,Philippe,7,"0,79","0,98",7,M,CHEMINADE,Jacques,1,"0,11","0,14",8,M,LASSALLE,Jean,9,"1,02","1,26",9,M,MÉLENCHON,Jean-Luc,113,"12,78","15,85",10,M,ASSELINEAU,François,3,"0,34","0,42",11,M,FILLON,François,189,"21,38","26,51",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47094,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0002,781,162,"20,74",619,"79,26",9,"1,15","1,45",3,"0,38","0,48",607,"77,72","98,06",1,M,DUPONT-AIGNAN,Nicolas,28,"3,59","4,61",2,F,LE PEN,Marine,140,"17,93","23,06",3,M,MACRON,Emmanuel,148,"18,95","24,38",4,M,HAMON,Benoît,28,"3,59","4,61",5,F,ARTHAUD,Nathalie,2,"0,26","0,33",6,M,POUTOU,Philippe,8,"1,02","1,32",7,M,CHEMINADE,Jacques,1,"0,13","0,16",8,M,LASSALLE,Jean,1,"0,13","0,16",9,M,MÉLENCHON,Jean-Luc,90,"11,52","14,83",10,M,ASSELINEAU,François,7,"0,90","1,15",11,M,FILLON,François,154,"19,72","25,37",https://www.data.gouv.fr/api/1/datasets/r/8fdb...,2026-03-18T14:25:34.183485,2017_burvot_t1_france_entiere.txt
47095,69,Rhône,09,9ème circonscription,004,Alix,0001,455,82,"18,02",373,"81,98",11,"2,42","2,95",2,"0,44","0,54",360,"79,12","96,51",1,M,DUPONT-AIGNAN,Nicolas,29,"6,37","8,06",2,F,LE PEN,Marine,88,"19,